# Referencia serial de Wiki Analytics — muestra local por cliente

Adaptación del notebook de Kaggle a los requisitos aclarados. Lee los Parquet locales y genera **Q1–Q5 para cada cliente por separado**, sin importar código Go. La ejecución de este notebook es un paso manual de preparación, nunca parte de la regresión automática.

- Q2: nombre, URL del artículo y bytes originales de imagen; ausencia explícita si no hay referencia.
- Q3: intersección de vocabularios por idioma, excluyendo stopwords y números.
- Q4: palabras distintas por abstract, top 20 con orden `(-count, word)`.
- Q5: ocurrencias, excluyendo stopwords y números; comparación exacta `count * N > S`.

Las listas NLTK inglesa/francesa están congeladas en `tests/oracle/resources/`. El sistema Go deberá usar esas mismas listas y el contrato de tokenización. Las imágenes pendientes o con HTTP 404 mantienen **Q2 y el paquete global incompletos**; no se reemplazan por resultados vacíos. Ver `tests/oracle/README.md`.

In [ ]:
from pathlib import Path
import os
import sys
import json

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "dataset").is_dir() and
               (path / "wikipedia-article-analysis.ipynb").is_file())
sys.path.insert(0, str(ROOT / "tests" / "oracle"))
import reference

CLIENTS = 2
# None usa las 250.000 filas; para un piloto elegir 100 o 1000 por archivo.
ROWS_PER_FILE = int(os.environ["WIKI_ORACLE_ROWS_PER_FILE"]) if os.environ.get("WIKI_ORACLE_ROWS_PER_FILE") else None
OUTPUT = Path(os.environ.get("WIKI_ORACLE_OUTPUT", ROOT / "tests" / "expected" / "full-5clients-v1"))
# Paso de red explícito; también puede ejecutarse después con capture-images.
DOWNLOAD_IMAGES = os.environ.get("WIKI_ORACLE_DOWNLOAD_IMAGES", "0") == "1"
DATASET = Path(os.environ["TINY_FIXTURES"])
print({"clients": CLIENTS, "rows_per_file": ROWS_PER_FILE,
       "output": str(OUTPUT), "download_images": DOWNLOAD_IMAGES})

## Contrato semántico y stopwords

`reference.py` contiene las operaciones seriales del notebook original y el soporte de entrada/salida. Python usa `str.lower()`, `re.findall(r"\b\w+\b", ...)` en Unicode y el filtro numérico original. No se aplica stemming ni eliminación de acentos. Q4 convierte los tokens de cada abstract en un conjunto; Q5 conserva las repeticiones. Abstract histórico nulo se excluye del denominador; vacío no nulo aporta cero y sí cuenta. Sin base histórica, Q5 queda vacío con estado explícito.

In [ ]:
STOPWORDS = reference.load_stopwords()
SEMANTICS = reference.contract()
assert reference.tokens("the alpha alpha et beta 123", STOPWORDS) == ["alpha", "alpha", "beta"]
print(json.dumps(SEMANTICS, ensure_ascii=False, indent=2))

## Entrada y partición reproducible

Se intercalan archivos `en`, `fr` por índice numérico. La posición global empieza en cero y la fila se asigna mediante `ordinal % CLIENTS`. Los índices no dependen del tamaño del lote. Cada manifiesto guarda rutas, hashes, esquema y cantidades; los Parquet permanecen en su ubicación original. El lector del oráculo proyecta campos analíticos; el cliente del sistema deberá transmitir registros completos.

Cada ocurrencia conserva identidad `(archivo, índice de fila)`: no se deduplican artículos repetidos. El presupuesto de la muestra completa es 467.363.817 bytes comprimidos, cercano a los 500 MB sugeridos.

In [ ]:
INPUTS = reference.inventory(DATASET, CLIENTS, ROWS_PER_FILE)
print(json.dumps({"files": len(INPUTS["files"]),
                  "selected_rows": INPUTS["selected_rows"],
                  "source_parquet_bytes": INPUTS["source_parquet_bytes"],
                  "rows_per_client": [len(range(c, INPUTS["selected_rows"], CLIENTS))
                                      for c in range(CLIENTS)]}, indent=2))

## Ejecutar las cinco consultas, un cliente a la vez

El destino debe ser nuevo: no se sobrescriben referencias ya generadas. Para cada cliente se recorre la entrada por lotes, manteniendo vocabularios, conteos y candidatos de ese cliente. No se carga el corpus completo ni se combinan estados entre clientes. Q5 espera su suma y denominador históricos definitivos.

Se exportan todas las filas a JSONL, sin truncamiento de presentación. Q1/Q2/Q5 conservan multiplicidades, Q3 tiene palabras únicas y Q4 tiene orden total. Q2 registra referencias pendientes hasta capturar sus bytes.

In [ ]:
MANIFEST = reference.create_bundle(OUTPUT, INPUTS, DATASET / "oracle.ipynb")
for client in range(CLIENTS):
    print(f"Procesando cliente {client}...", flush=True)
    results, stats = reference.serial_queries(
        reference.client_rows(DATASET, INPUTS, client), STOPWORDS)
    entry = reference.export_client(OUTPUT, MANIFEST, client, results, stats)
    print(json.dumps({"client": entry["id"], "input_rows": entry["input_rows"],
                      "languages": entry["languages"],
                      "historical_sum": entry["historical_sum"],
                      "historical_count": entry["historical_count"],
                      "query_rows": {q: m["rows"] for q, m in entry["queries"].items()}},
                     ensure_ascii=False), flush=True)
    del results
reference.finish_bundle(OUTPUT, MANIFEST)

## Captura opcional de imágenes originales de Q2

El oráculo descarga la URL `image.content_url` almacenada en el Parquet, sin sustituirla por miniaturas o por la imagen actual de otro artículo. El fetcher del sistema puede resolver metadatos mediante la API; las regresiones deberán usar los bytes congelados aquí.

Una URL se descarga una sola vez en esta captura serial. Se guardan bytes, SHA-256, longitud, URL resuelta y cabeceras HTTP. Los HTTP 404 se registran como `not_found` y la captura continúa con las demás URLs. Otros errores detienen la captura sin reintentos automáticos. Q2 sigue incompleta mientras falten bytes, incluidos los 404. Un nuevo intento manual conserva los bytes ya capturados. Esto es preparación de referencias, no recuperación de ejecuciones del sistema.

In [ ]:
if DOWNLOAD_IMAGES:
    try:
        reference.capture_images(OUTPUT)
    except (OSError, ValueError, RuntimeError) as error:
        print(f"CAPTURA INCOMPLETA: {error}")
else:
    print("Captura de red no solicitada. Ejecutar capture-images para completar Q2.")

In [ ]:
import shutil
blob = DATASET / "services" / "pixel.png"
sha = reference.digest(blob)
shared = OUTPUT.parent / "images"
shared.mkdir(exist_ok=True)
target = shared / f"{sha}.bin"
if target.exists():
    assert reference.digest(target) == sha
else:
    shutil.copy2(blob, target)
image = {"source_url": "https://images.tiny.invalid/pixel.png", "status": "available",
         "path": f"../images/{sha}.bin", "sha256": sha, "bytes": blob.stat().st_size,
         "provenance": "synthetic_fixture"}
index = reference.load_image_index(shared)
assert image["source_url"] not in index or index[image["source_url"]] == image
index[image["source_url"]] = image
reference.write_image_index(shared, index)
for client in MANIFEST["clients"]:
    info = client["queries"]["q2"]
    path = OUTPUT / info["path"]
    records = reference.read_jsonl(path)
    for row in records:
        if row["image"] is not None:
            assert row["image"]["source_url"] == image["source_url"]
            row["image"] = dict(image)
    reference.write_jsonl(path, records)
    info["sha256"] = reference.digest(path)
    info["status"] = "complete"
MANIFEST["scenario"] = "tiny-v1"
MANIFEST["synthetic"] = True
MANIFEST["fixture_files"] = {str(p.relative_to(DATASET)): reference.digest(p)
    for p in sorted(DATASET.rglob("*")) if p.is_file()}
MANIFEST["integration"] = {"status": "pending", "required_queries": [1, 2, 3, 4, 5],
    "blockers": ["Ingesta completa y captura Q1–Q5 del cliente oficial",
                 "Disponibilidad y cierres de todas las etapas Q1–Q5",
                 "Agregación de los cinco cierres en ResultCollector",
                 "Endpoint local de metadatos/contenido compatible con ImageFetcher",
                 "Evidencia de solapamiento y formato de captura de cierres"]}
recipe = ROOT / "tests" / "integration" / "prepare_tiny.py"
shutil.copy2(recipe, OUTPUT / "generator" / "prepare_tiny.py")
MANIFEST["generator"]["preparation_snapshot_path"] = "generator/prepare_tiny.py"
MANIFEST["generator"]["preparation_sha256"] = reference.digest(recipe)
reference.finish_bundle(OUTPUT, MANIFEST)


## Auditoría y estado final

La auditoría comprueba hashes y cantidades de los archivos exportados. No reemplaza la comparación contra el sistema distribuido. El comando `reference.py verify` exige por defecto todas las imágenes y devuelve error si faltan. Sólo `status == "complete"` habilita el paquete entero como referencia Q1–Q5.

In [ ]:
AUDIT = reference.verify_bundle(OUTPUT, DATASET, require_images=False)
print(json.dumps(AUDIT, ensure_ascii=False, indent=2))
if AUDIT["status"] != "complete":
    print("ATENCIÓN: Q1/Q3/Q4/Q5 exportadas; Q2 pendiente de bytes. No validar Q2 como exitosa.")
else:
    print("Referencias Q1–Q5 completas. Conservar este paquete y sus imágenes.")